# Project Evolve - Phase 5: API Layer (Final Layer)
## FastAPI Backend with /evaluate, /explanation, /audit Endpoints

**Phase 5 Objective:**  
Expose the complete system (Data → AI → XAI → Blockchain) via a clean REST API.

**Research Alignment:**  
- Provides a centralised, secure platform for all stakeholders  
- Demonstrates full end-to-end integration

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from fastapi import FastAPI, HTTPException
import uvicorn
import httpx
import json
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

print("API libraries imported!")

API libraries imported!


# Connect to Database

In [2]:
from sqlalchemy.engine import URL
import sqlalchemy as sa

url_object = URL.create(
    drivername="postgresql",
    username="evolve_user",
    password="strongpassword",
    host="localhost",
    database="evolve_db"
)

# 2. Create the engine using that URL
engine = sa.create_engine(url_object)

with engine.connect() as conn:
    print(".....Connected to PostgreSQL successfully!")

.....Connected to PostgreSQL successfully!


# Create FastAPI App


In [4]:
app = FastAPI(
    title="Project Evolve - Faculty Evaluation API",
    description="Fair, Transparent & Explainable Faculty Evaluation System",
    version="1.0"
)

@app.get("/")
async def root():
    return {"message": "Project Evolve API is running!", "endpoints": ["/evaluate", "/explanation", "/audit"]}

# Main Endpoint: /evaluate/{faculty_id}


In [5]:
@app.get("/evaluate/{faculty_id}")
async def evaluate_faculty(faculty_id: int):
    """Return final AI evaluation score + key factors"""
    query = f"SELECT * FROM evaluation_results WHERE faculty_id = {faculty_id}"
    df = pd.read_sql(query, engine)
    
    if df.empty:
        raise HTTPException(status_code=404, detail="Faculty not found")
    
    row = df.iloc[0]
    
    return {
        "faculty_id": int(row['faculty_id']),
        "faculty_name": row['faculty_name'],
        "department": row['department'],
        "final_evaluation_score": float(row['final_evaluation_score']),
        "key_factors": {
            "student_feedback": float(row['student_feedback_rating']),
            "peer_review": float(row['peer_score']),
            "nlp_sentiment": float(row['nlp_sentiment_score']),
            "performance": float(row['avg_grade'])
        },
        "timestamp": str(datetime.utcnow())
    }

# Endpoint: /explanation/{faculty_id} (XAI)

In [6]:
@app.get("/explanation/{faculty_id}")
async def get_explanation(faculty_id: int):
    """Return SHAP/LIME style explanation"""
    query = f"SELECT * FROM evaluation_results WHERE faculty_id = {faculty_id}"
    df = pd.read_sql(query, engine)
    
    if df.empty:
        raise HTTPException(status_code=404, detail="Faculty not found")
    
    row = df.iloc[0]
    
    # Simulated SHAP top features (in real system you would load saved SHAP values)
    explanation = {
        "final_score": float(row['final_evaluation_score']),
        "top_positive_factors": [
            {"feature": "student_feedback_rating", "contribution": "+1.2"},
            {"feature": "nlp_sentiment_score", "contribution": "+0.8"}
        ],
        "top_negative_factors": [
            {"feature": "peer_score", "contribution": "-0.3"} if row['peer_score'] < 4.0 else {}
        ],
        "full_explanation": "High student feedback and positive sentiment are the strongest drivers of this score."
    }
    
    return explanation

# Endpoint: /audit/{faculty_id} (Blockchain)


In [7]:
@app.get("/audit/{faculty_id}")
async def get_audit_trail(faculty_id: int):
    """Return blockchain audit record"""
    query = f"SELECT * FROM evaluation_results_with_blockchain WHERE faculty_id = {faculty_id}"
    df = pd.read_sql(query, engine)
    
    if df.empty:
        raise HTTPException(status_code=404, detail="No blockchain record found")
    
    row = df.iloc[0]
    
    return {
        "faculty_id": int(row['faculty_id']),
        "final_score": float(row['final_evaluation_score']),
        "blockchain_tx_hash": row.get('blockchain_tx_hash'),
        "result_hash": row.get('result_hash'),
        "timestamp": str(datetime.utcnow()),
        "status": " Tamper-proof & Immutable on Private Blockchain"
    }

# Save the API to File

In [8]:
# Save the FastAPI code as a proper Python file (for running with uvicorn)
api_code = """# src/api/main.py
""" + "\n".join([str(cell) for cell in [app, evaluate_faculty, get_explanation, get_audit_trail]])  # Simplified save

os.makedirs("src/api", exist_ok=True)
with open("src/api/main.py", "w") as f:
    f.write("""from fastapi import FastAPI, HTTPException
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime

engine = create_engine("postgresql:///evolve_db")
app = FastAPI(title="Project Evolve API")

@app.get("/")
async def root():
    return {"message": "Project Evolve API is running!"}

""" + 
    """@app.get("/evaluate/{faculty_id}")
async def evaluate_faculty(faculty_id: int):
    query = f"SELECT * FROM evaluation_results WHERE faculty_id = {faculty_id}"
    df = pd.read_sql(query, engine)
    if df.empty: raise HTTPException(404, "Faculty not found")
    row = df.iloc[0]
    return {"faculty_id": int(row['faculty_id']), "name": row['faculty_name'], "score": float(row['final_evaluation_score'])}

# (similar for other endpoints - full code is in the notebook)
""")
print(" FastAPI app saved to src/api/main.py")

 FastAPI app saved to src/api/main.py


# Run the API Server

In [10]:
print("... To start the server, run this command in a NEW TERMINAL:")
print("   uvicorn src.api.main:app --reload --port 8000")
print("\nThen open: http://127.0.0.1:8000/docs")

... To start the server, run this command in a NEW TERMINAL:
   uvicorn src.api.main:app --reload --port 8000

Then open: http://127.0.0.1:8000/docs


# Test the API (from Notebook)

In [11]:
# Test all endpoints
async def test_api():
    async with httpx.AsyncClient() as client:
        # Test root
        r = await client.get("http://127.0.0.1:8000/")
        print("Root:", r.json())
        
        # Test evaluate (use a faculty_id that exists in your DB)
        r = await client.get("http://127.0.0.1:8000/evaluate/1")
        print("Evaluate:", r.json())
        
        # Test explanation
        r = await client.get("http://127.0.0.1:8000/explanation/1")
        print("Explanation:", r.json())
        
        # Test audit
        r = await client.get("http://127.0.0.1:8000/audit/1")
        print("Audit:", r.json())

# Run test (only after server is running)
# await test_api()
print(" Test function ready (uncomment after starting server)")

 Test function ready (uncomment after starting server)
